### Context Aware Music Recommender System

##### Imports

In [60]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix

### Read the datas

##### Last.fm

In [61]:

### last.fm data
columns = ['userid', 'timestamp', 'artid', 'artname', 'traid', 'traname']

df_lastfm = pd.read_csv(
        'data/lastfm-dataset-1K/userid-timestamp-artid-artname-traid-traname.tsv', 
        sep='\t', 
        on_bad_lines='skip',
        names=columns, 
        header=None,
        nrows=100000 # full load is 19 millions rows, cap it to 100,000
    )
df_lastfm


,userid,timestamp,artid,artname,traid,traname
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15)
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15)
3,user_000001,2009-05-04T13:42:52Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Hibari (Live_2009_4_15)
4,user_000001,2009-05-04T13:42:11Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc1 (Live_2009_4_15)
...,...,...,...,...,...,...
99995,user_000004,2008-03-20T14:08:49Z,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,9c7da1c0-108a-456a-baa2-e5d29e349324,Get Lucky
99996,user_000004,2008-03-20T14:04:57Z,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,27534cc1-6140-4c0a-82e2-dfc1b32bcdea,Tight Fit
99997,user_000004,2008-03-20T13:59:30Z,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,c271d600-46db-4f65-9231-dd5d334b12c8,Get Dancey
99998,user_000004,2008-03-20T13:55:22Z,61cd6f2a-cdde-4f34-aacd-b992badcacba,New Young Pony Club,9c7732f4-40e2-40ab-b113-b3b73bab51b5,Jerk Me


In [62]:
df_lastfm.shape

(100000, 6)

In [63]:
df_lastfm.columns

Index(['userid', 'timestamp', 'artid', 'artname', 'traid', 'traname'], dtype='str')

##### Spotify Data

In [64]:
df_spotify = pd.read_csv('data/spotify-dataset/data.csv')  # kaggle 
df_spotify


,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
0,0.0594,1921,0.98200,"['Sergei Rachmaninoff', 'James Levine', 'Berli...",0.279,831667,0.211,0,4BJqT0PrAfrxzMOxytFOIz,0.878000,10,0.6650,-20.096,1,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",4,1921,0.0366,80.954
1,0.9630,1921,0.73200,['Dennis Day'],0.819,180533,0.341,0,7xPhfUan2yNtyFG0cUWkt8,0.000000,7,0.1600,-12.441,1,Clancy Lowered the Boom,5,1921,0.4150,60.936
2,0.0394,1921,0.96100,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,0.328,500062,0.166,0,1o6I8BglA6ylDMrIELygv1,0.913000,3,0.1010,-14.850,1,Gati Bali,5,1921,0.0339,110.339
3,0.1650,1921,0.96700,['Frank Parker'],0.275,210000,0.309,0,3ftBPsC5vPBKxYSee08FDH,0.000028,5,0.3810,-9.316,1,Danny Boy,3,1921,0.0354,100.109
4,0.2530,1921,0.95700,['Phil Regan'],0.418,166693,0.193,0,4d6HGyGT8e121BsdKmw9v6,0.000002,3,0.2290,-10.096,1,When Irish Eyes Are Smiling,2,1921,0.0380,101.665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170648,0.6080,2020,0.08460,"['Anuel AA', 'Daddy Yankee', 'KAROL G', 'Ozuna...",0.786,301714,0.808,0,0KkIkfsLEJbrcIhYsCL7L5,0.000289,7,0.0822,-3.702,1,China,72,2020-05-29,0.0881,105.029
170649,0.7340,2020,0.20600,['Ashnikko'],0.717,150654,0.753,0,0OStKKAuXlxA0fMH54Qs6E,0.000000,7,0.1010,-6.020,1,Halloweenie III: Seven Days,68,2020-10-23,0.0605,137.936
170650,0.6370,2020,0.10100,['MAMAMOO'],0.634,211280,0.858,0,4BZXVFYCb76Q0Klojq4piV,0.000009,4,0.2580,-2.226,0,AYA,76,2020-11-03,0.0809,91.688
170651,0.1950,2020,0.00998,['Eminem'],0.671,337147,0.623,1,5SiZJoLXp3WOl3J4C8IK0d,0.000008,2,0.6430,-7.161,1,Darkness,70,2020-01-17,0.3080,75.055


### Feature Extraction

In [65]:
# extract datetime
df_lastfm["datetime"] = pd.to_datetime(df_lastfm["timestamp"], utc=True)
df_lastfm.head(3)

,userid,timestamp,artid,artname,traid,traname,datetime
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007,2009-05-04 23:08:57+00:00
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15),2009-05-04 13:54:10+00:00
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15),2009-05-04 13:52:04+00:00


In [66]:
# get hour and time_of_day
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    else:
        return "night"

df_lastfm["hour"] = df_lastfm["datetime"].dt.hour
df_lastfm["time_of_day"] = df_lastfm["hour"].apply(get_time_of_day)
df_lastfm.head(3)

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007,2009-05-04 23:08:57+00:00,23,night
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15),2009-05-04 13:54:10+00:00,13,afternoon
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15),2009-05-04 13:52:04+00:00,13,afternoon


In [67]:
# extract weekday & weekend
df_lastfm["day_of_week"] = df_lastfm["datetime"].dt.weekday
df_lastfm["day_type"] = df_lastfm["day_of_week"].apply(lambda x: "weekend" if x >= 5 else "weekday")
df_lastfm.head(3)

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day,day_of_week,day_type
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007,2009-05-04 23:08:57+00:00,23,night,0,weekday
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15),2009-05-04 13:54:10+00:00,13,afternoon,0,weekday
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15),2009-05-04 13:52:04+00:00,13,afternoon,0,weekday


In [68]:
# check if weekend is extracted
df_lastfm[df_lastfm['day_type'] == 'weekend'].head()

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day,day_of_week,day_type
14,user_000001,2009-05-03T15:48:25Z,ba2f4f3b-0293-4bc8-bb94-2f73b5207343,Underworld,dc394163-2b78-4b56-94e4-658597a29ef8,"Boy, Boy, Boy (Switch Remix)",2009-05-03 15:48:25+00:00,15,afternoon,6,weekend
15,user_000001,2009-05-03T15:37:56Z,ba2f4f3b-0293-4bc8-bb94-2f73b5207343,Underworld,340d9a0b-9a43-4098-b116-9f79811bd508,Crocodile (Innervisions Orchestra Mix),2009-05-03 15:37:56+00:00,15,afternoon,6,weekend
16,user_000001,2009-05-03T15:14:53Z,a16e47f5-aa54-47fe-87e4-bb8af91a9fdd,Ennio Morricone,0b04407b-f517-4e00-9e6a-494795efc73e,Ninna Nanna In Blu (Raw Deal Remix),2009-05-03 15:14:53+00:00,15,afternoon,6,weekend
17,user_000001,2009-05-03T15:10:18Z,463a94f1-2713-40b1-9c88-dcc9c0170cae,Minus 8,4e78efc4-e545-47af-9617-05ff816d86e2,Elysian Fields,2009-05-03 15:10:18+00:00,15,afternoon,6,weekend
18,user_000001,2009-05-03T15:04:31Z,ad0811ea-e213-451d-b22f-fa1a7f9e0226,Beanfield,fb51d2c4-cc69-4128-92f5-77ec38d66859,Planetary Deadlock,2009-05-03 15:04:31+00:00,15,afternoon,6,weekend


##### Listening sessions
A session groups songs played close in time.
```
If gap between songs > 30 minutes → new session
```

In [69]:
# extract listening session

df_lastfm = df_lastfm.sort_values(['userid', 'datetime'])
df_lastfm["time_diff"] = df_lastfm.groupby("userid")["datetime"].diff().dt.seconds # in seconds

SESSION_GAP = 30 * 60 # in seconds

df_lastfm["new_session"] = (df_lastfm["time_diff"] > SESSION_GAP) | (df_lastfm["time_diff"].isna())

df_lastfm['session_id'] = df_lastfm.groupby("userid")["new_session"].cumsum()

df_lastfm.head(100)

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day,day_of_week,day_type,time_diff,new_session,session_id
16684,user_000001,2006-08-13T13:59:20Z,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,c4633ab1-e715-477f-8685-afa5f2058e42,The Launching Of Big Face,2006-08-13 13:59:20+00:00,13,afternoon,6,weekend,NaN,True,1
16683,user_000001,2006-08-13T14:03:29Z,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,bc2765af-208c-44c5-b3b0-cf597a646660,Zn Zero,2006-08-13 14:03:29+00:00,14,afternoon,6,weekend,249.0,False,1
16682,user_000001,2006-08-13T14:10:43Z,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,aa9c5a80-5cbe-42aa-a966-eb3cfa37d832,The Return Of Super Barrio - End Credits,2006-08-13 14:10:43+00:00,14,afternoon,6,weekend,434.0,False,1
16681,user_000001,2006-08-13T14:17:40Z,67fb65b5-6589-47f0-9371-8a40eb268dfb,Tommy Guerrero,d9b1c1da-7e47-4f97-a135-77260f2f559d,Mission Flats,2006-08-13 14:17:40+00:00,14,afternoon,6,weekend,417.0,False,1
16680,user_000001,2006-08-13T14:19:06Z,1cfbc7d1-299c-46e6-ba4c-1facb84ba435,Artful Dodger,120bb01c-03e4-465f-94a0-dce5e9fac711,What You Gonna Do?,2006-08-13 14:19:06+00:00,14,afternoon,6,weekend,86.0,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16589,user_000001,2006-08-16T11:10:38Z,07729ca8-b15b-44d4-881b-6a325f61665a,Maysa,d1bdae20-9ba3-4c49-ae3c-99089cdd6d1a,Blue Horizon,2006-08-16 11:10:38+00:00,11,morning,2,weekday,62.0,False,3
16588,user_000001,2006-08-16T11:14:54Z,07729ca8-b15b-44d4-881b-6a325f61665a,Maysa,bbb904d2-d28b-4ecf-933a-6aa6720d71de,Osaka,2006-08-16 11:14:54+00:00,11,morning,2,weekday,256.0,False,3
16587,user_000001,2006-08-16T11:16:34Z,07729ca8-b15b-44d4-881b-6a325f61665a,Maysa,b37a264d-4f4d-4289-b2c4-8a9dc102171d,Everything,2006-08-16 11:16:34+00:00,11,morning,2,weekday,100.0,False,3
16586,user_000001,2006-08-16T13:43:02Z,07729ca8-b15b-44d4-881b-6a325f61665a,Maysa,4d201e6a-e823-4ed0-80a3-ebc32492c369,Tailor Made Love,2006-08-16 13:43:02+00:00,13,afternoon,2,weekday,8788.0,True,4


##### User taste
User taste means what artists or songs the user prefers.
```
play_count
```

In [70]:
# extract user prefer artist
user_artist_pref = (
    df_lastfm.groupby(["userid", "artname"])
      .size()
      .reset_index(name="play_count")
)
user_artist_pref.sort_values("play_count", ascending=False) # top artist

,userid,artname,play_count
1738,user_000002,The Libertines,2411
762,user_000002,Babyshambles,1724
1235,user_000002,Kettcar,1282
1734,user_000002,The Kooks,978
1315,user_000002,Maxïmo Park,941
...,...,...,...
3829,user_000004,Xiu Xiu,1
3831,user_000004,Yael Naïm,1
3833,user_000004,Yelle,1
3834,user_000004,Yo La Tengo,1


#### Feature extraction summary
From the original columns you can derive:

From `timestamp`

- time_of_day
- weekday/weekend
- hour
- month
- season

From `timestamp gaps`

- session_id
- session_length
- session_position

From `listening history`

- user favorite artists
- user favorite songs
- artist popularity

## Recommendation Engine

#### Merge both dataset

In [71]:

def normalise(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9 ]", "", s)   # strip punctuation
    s = re.sub(r"\s+", " ", s)
    return s

df_spotify['name_norm']    = df_spotify['name'].apply(normalise)
df_spotify['artists_norm'] = df_spotify['artists'].apply(normalise)

df_lastfm['traname_norm'] = df_lastfm['traname'].apply(normalise)
df_lastfm['artname_norm'] = df_lastfm['artname'].apply(normalise)

# exact merge on both columns
merged = df_lastfm.merge(
    df_spotify,
    left_on=['artname_norm', 'traname_norm'],
    right_on=['artists_norm', 'name_norm'],
    how='left'
)

match_rate = merged['tempo'].notna().mean()
print(f"Match rate: {match_rate:.1%}")

Match rate: 64.1%


In [72]:
merged.columns

Index(['userid', 'timestamp', 'artid', 'artname', 'traid', 'traname',
       'datetime', 'hour', 'time_of_day', 'day_of_week', 'day_type',
       'time_diff', 'new_session', 'session_id', 'traname_norm',
       'artname_norm', 'valence', 'year', 'acousticness', 'artists',
       'danceability', 'duration_ms', 'energy', 'explicit', 'id',
       'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'name',
       'popularity', 'release_date', 'speechiness', 'tempo', 'name_norm',
       'artists_norm'],
      dtype='str')

In [73]:
merged['liveness'].notna().sum()

np.int64(137194)

In [74]:
merged[merged['liveness'].notna()].head()

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day,day_of_week,...,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo,name_norm,artists_norm
8,user_000001,2006-08-13T14:59:59Z,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,1c0377bb-c00b-4bbe-b4b2-615f13324adc,Idioteque,2006-08-13 14:59:59+00:00,14,afternoon,6,...,0.0914,-7.800,1.0,Idioteque,59.0,2000-10-01,0.2400,137.544,idioteque,radiohead
17,user_000001,2006-08-13T15:44:17Z,69158f97-4c07-4c4e-baf8-4e4ab1ed666e,Boards Of Canada,ef756a43-0905-4e59-8f05-a6b2bd977aa8,Dayvan Cowboy,2006-08-13 15:44:17+00:00,15,afternoon,6,...,0.0677,-10.774,1.0,Dayvan Cowboy,55.0,2005-10-17,0.0427,167.950,dayvan cowboy,boards of canada
25,user_000001,2006-08-13T16:24:47Z,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,31c89fdc-afdb-4d84-bc61-2dbabf0b4f23,Treefingers,2006-08-13 16:24:47+00:00,16,afternoon,6,...,0.1090,-21.359,1.0,Treefingers,63.0,2000-10-01,0.0354,138.305,treefingers,radiohead
33,user_000001,2006-08-13T16:57:01Z,69158f97-4c07-4c4e-baf8-4e4ab1ed666e,Boards Of Canada,93fa1eb2-dd7c-4e0e-8174-360d32176586,Peacock Tail,2006-08-13 16:57:01+00:00,16,afternoon,6,...,0.0788,-13.083,1.0,Peacock Tail,52.0,2005-10-17,0.0580,171.424,peacock tail,boards of canada
36,user_000001,2006-08-13T17:09:53Z,69158f97-4c07-4c4e-baf8-4e4ab1ed666e,Boards Of Canada,ef756a43-0905-4e59-8f05-a6b2bd977aa8,Dayvan Cowboy,2006-08-13 17:09:53+00:00,17,evening,6,...,0.0677,-10.774,1.0,Dayvan Cowboy,55.0,2005-10-17,0.0427,167.950,dayvan cowboy,boards of canada


###  Build the track item matrix
one row per unique (artname, traname) with context features

In [75]:
AUDIO_FEATURES = [
    'acousticness', 'danceability', 'energy',
    'instrumentalness', 'loudness', 'speechiness',
    'tempo', 'valence', 'popularity'
]
TIME_SLOTS = ['morning', 'afternoon', 'evening', 'night']

# unique tracks enriched with spotify features
tracks = merged.groupby(['artname', 'traname']).agg(
    total_plays=('userid', 'count'),
    **{f: (f, 'mean') for f in AUDIO_FEATURES}   # avg if multiple rows
).reset_index()

# --- fill missing audio features with artist-level mean ---
for feat in AUDIO_FEATURES:
    artist_means = tracks.groupby('artname')[feat].transform('mean')
    tracks[feat] = tracks[feat].fillna(artist_means)
    tracks[feat] = tracks[feat].fillna(tracks[feat].median())  # global fallback

# --- scale audio features to [0, 1] ---
scaler = MinMaxScaler()
tracks[AUDIO_FEATURES] = scaler.fit_transform(tracks[AUDIO_FEATURES])

# --- time-of-day distribution from Last.fm ---
tod_counts = (
    merged.groupby(['artname', 'traname', 'time_of_day'])
    .size().unstack(fill_value=0)
    .reindex(columns=TIME_SLOTS, fill_value=0)
)
tod_counts = tod_counts.div(tod_counts.sum(axis=1).replace(0, 1), axis=0)
tracks = tracks.merge(tod_counts.reset_index(), on=['artname', 'traname'], how='left')
tracks[TIME_SLOTS] = tracks[TIME_SLOTS].fillna(0.25)

tracks['track_id'] = range(len(tracks))

# --- final feature matrix: audio + time-of-day ---
from scipy.sparse import csr_matrix
audio_matrix = csr_matrix(tracks[AUDIO_FEATURES].values)
tod_matrix   = csr_matrix(tracks[TIME_SLOTS].values)
item_matrix  = hstack([audio_matrix, tod_matrix])

In [76]:
tracks

,artname,traname,total_plays,acousticness,danceability,energy,instrumentalness,loudness,speechiness,tempo,valence,popularity,morning,afternoon,evening,night,track_id
0,!!!,A New Name,4,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,0.500091,0.545455,0.000000,1.000000,0.000000,0.0,0
1,!!!,All My Heroes Are Weirdos,4,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,0.500091,0.545455,0.000000,1.000000,0.000000,0.0,1
2,!!!,Bend Over Beethoven,4,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,0.500091,0.545455,0.000000,0.750000,0.250000,0.0,2
3,!!!,Break In Case Of Anything,3,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,0.500091,0.545455,0.000000,1.000000,0.000000,0.0,3
4,!!!,Hammerhead,1,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,0.500091,0.545455,0.000000,0.000000,0.000000,1.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18002,高木正勝,Private Drawing,3,0.617469,0.271889,0.632463,0.408537,0.703815,0.056992,0.200667,0.382612,0.519481,0.000000,1.000000,0.000000,0.0,18002
18003,高木正勝,Shing Morno,7,0.617469,0.271889,0.632463,0.408537,0.703815,0.056992,0.200667,0.382612,0.519481,0.142857,0.571429,0.285714,0.0,18003
18004,高木正勝,Trinidad Bird,10,0.617469,0.271889,0.632463,0.408537,0.703815,0.056992,0.200667,0.382612,0.519481,0.100000,0.800000,0.100000,0.0,18004
18005,高木正勝,Venetian Red,10,0.617469,0.271889,0.632463,0.408537,0.703815,0.056992,0.200667,0.382612,0.519481,0.100000,0.700000,0.200000,0.0,18005


### Build the user profile vector
weighted avg of all tracks the user played -> one vector per user


In [77]:
def build_user_profile(userid, df, tracks, item_matrix):
    user_plays = (
        df[df['userid'] == userid]
        .groupby(['artname', 'traname'])
        .size().reset_index(name='play_count')
    )
    user_plays = user_plays.merge(
        tracks[['artname', 'traname', 'track_id']],
        on=['artname', 'traname'], how='inner'
    )
    if user_plays.empty:
        return None, None

    weights = user_plays['play_count'].values.astype(float)
    weights /= weights.sum()
    profile = csr_matrix(weights) @ item_matrix[user_plays['track_id'].values]
    return profile, set(zip(user_plays['artname'], user_plays['traname']))

### Candidate pool
track that the user has **not** heard

In [78]:
def get_candidates(userid, df, tracks):
    heard = set(zip(
        df[df['userid'] == userid]['artname'],
        df[df['userid'] == userid]['traname']
    ))
    candidates = tracks[~tracks.apply(
        lambda r: (r['artname'], r['traname']) in heard, axis=1
    )]
    return candidates

### Cosine similarity
score every candidate against the user profile vector


In [79]:
from sklearn.metrics.pairwise import cosine_similarity

def score_candidates(user_profile, candidates, item_matrix):
    cand_vectors = item_matrix[candidates['track_id'].values]
    scores = cosine_similarity(user_profile, cand_vectors).flatten()
    candidates = candidates.copy()
    candidates['base_score'] = scores
    return candidates

### Context boost
multiply score if track's typical time_of_day matches current session


In [80]:
BOOST = 1.25  # 25% lift

def apply_context_boost(candidates, current_time_of_day):
    candidates = candidates.copy()
    # dominant TOD for each candidate track
    candidates['dominant_tod'] = candidates[TIME_SLOTS].idxmax(axis=1)
    match = candidates['dominant_tod'] == current_time_of_day
    candidates['final_score'] = candidates['base_score'] * np.where(match, BOOST, 1.0)
    return candidates

### Rank and return top n
sort by boosted score, return top 5–10 with explanation string


In [81]:
def recommend(userid, current_time_of_day, df, tracks, item_matrix, n=10):
    profile, _ = build_user_profile(userid, df, tracks, item_matrix)
    if profile is None:
        return pd.DataFrame()

    candidates = get_candidates(userid, df, tracks)
    candidates = score_candidates(profile, candidates, item_matrix)
    candidates = apply_context_boost(candidates, current_time_of_day)
    top_n = candidates.sort_values('final_score', ascending=False).head(n)
    return top_n

### Rule base explain
generate "why" string from the top contributing feature


In [82]:
# Settings to use
USER = 'user_000001'
CURRENT_TIME_OF_DAY = 'night'

FEATURE_LABELS = {
    'energy':          'high energy',
    'acousticness':    'acoustic feel',
    'danceability':    'danceability',
    'valence':         'positive mood',
    'instrumentalness':'instrumental style',
    'tempo':           'tempo',
}

In [ ]:
# define explain() to use correct signature
def explain(row, user_means, current_time_of_day):
    diffs = {f: abs(row[f] - user_means[f]) for f in FEATURE_LABELS}
    top_feature = min(diffs, key=diffs.get)
    label = FEATURE_LABELS[top_feature]

    parts = [f"Matches your preferred {label}"]
    if row['dominant_tod'] == current_time_of_day:
        parts.append(f"fits your {current_time_of_day} listening pattern")
    if row['total_plays'] > tracks['total_plays'].quantile(0.75):
        parts.append("popular track overall")
    return " · ".join(parts)

In [87]:
# Step 1 — compute user's mean for each audio feature (do this ONCE, outside apply)
user_heard = df_lastfm[df_lastfm['userid'] == USER][['artname', 'traname']]
user_tracks = user_heard.merge(tracks, on=['artname', 'traname'], how='inner')

# guard agaist user_tracks empty
if user_tracks.empty or user_tracks[list(FEATURE_LABELS)].isnull().all().all():
    print(f"No enriched tracks found for user — check merge step")
else:
    user_means = {f: user_tracks[f].mean() for f in FEATURE_LABELS}
# user_means is now e.g. {'energy': 0.62, 'acousticness': 0.18, ...}

# Step 2 — call recommend & explain
results = recommend(USER, CURRENT_TIME_OF_DAY, df_lastfm, tracks, item_matrix)
results['explanation'] = results.apply(
    lambda r: explain(r, user_means, CURRENT_TIME_OF_DAY), axis=1
)

print(results[['artname', 'traname', 'final_score', 'explanation']].to_string())

                  artname                               traname  final_score                                                                                       explanation
9099          Maxïmo Park                          Our Velocity     1.173675  Matches your preferred positive mood · fits your night listening pattern · popular track overall
12293     Shout Out Louds                        Oh, Sweetheart     1.171158  Matches your preferred positive mood · fits your night listening pattern · popular track overall
838              Ane Brun                        The Fight Song     1.169722  Matches your preferred positive mood · fits your night listening pattern · popular track overall
9089          Maxïmo Park                              Graffiti     1.169084  Matches your preferred positive mood · fits your night listening pattern · popular track overall
1478         Babyshambles                           Pentonville     1.168307  Matches your preferred positive mood · fits you

In [85]:
user_heard

,artname,traname
16684,Plaid & Bob Jaroc,The Launching Of Big Face
16683,Plaid & Bob Jaroc,Zn Zero
16682,Plaid & Bob Jaroc,The Return Of Super Barrio - End Credits
16681,Tommy Guerrero,Mission Flats
16680,Artful Dodger,What You Gonna Do?
...,...,...
4,坂本龍一,Mc1 (Live_2009_4_15)
3,坂本龍一,Hibari (Live_2009_4_15)
2,坂本龍一,Mc2 (Live_2009_4_15)
1,坂本龍一,Composition 0919 (Live_2009_4_15)


In [86]:
results

,artname,traname,total_plays,acousticness,danceability,energy,instrumentalness,loudness,speechiness,tempo,...,popularity,morning,afternoon,evening,night,track_id,base_score,dominant_tod,final_score,explanation
9099,Maxïmo Park,Our Velocity,26,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,...,0.545455,0.115385,0.346154,0.153846,0.384615,9099,0.938940,night,1.173675,Matches your preferred positive mood · fits yo...
12293,Shout Out Louds,"Oh, Sweetheart",17,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,...,0.545455,0.176471,0.352941,0.058824,0.411765,12293,0.936926,night,1.171158,Matches your preferred positive mood · fits yo...
838,Ane Brun,The Fight Song,18,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,...,0.545455,0.111111,0.333333,0.166667,0.388889,838,0.935777,night,1.169722,Matches your preferred positive mood · fits yo...
9089,Maxïmo Park,Graffiti,56,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,...,0.545455,0.232143,0.303571,0.125000,0.339286,9089,0.935268,night,1.169084,Matches your preferred positive mood · fits yo...
1478,Babyshambles,Pentonville,62,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,...,0.545455,0.193548,0.290323,0.209677,0.306452,1478,0.934646,night,1.168307,Matches your preferred positive mood · fits yo...
1672,Bedouin Soundclash,Money Worries (Feat. Vernon Maytone),24,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,...,0.545455,0.208333,0.291667,0.166667,0.333333,1672,0.933503,night,1.166879,Matches your preferred positive mood · fits yo...
7903,Kettcar,Balu,94,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,...,0.545455,0.180851,0.287234,0.212766,0.319149,7903,0.932920,night,1.166150,Matches your preferred positive mood · fits yo...
16837,Two Gallants,Nothing To You,11,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,...,0.545455,0.090909,0.363636,0.090909,0.454545,16837,0.932817,night,1.166022,Matches your preferred positive mood · fits yo...
15551,The Shins,We Will Become Silhouettes,12,0.285862,0.497607,0.589051,0.021400,0.775125,0.024259,0.411637,...,0.544456,0.083333,0.333333,0.166667,0.416667,15551,0.932507,night,1.165634,Matches your preferred positive mood · fits yo...
16244,The Zutons,Remember Me,16,0.130771,0.457565,0.698310,0.002042,0.786626,0.042425,0.440689,...,0.545455,0.250000,0.312500,0.062500,0.375000,16244,0.932217,night,1.165271,Matches your preferred positive mood · fits yo...
